# Hugging Face Local Pipelines 심화


> 업데이트 기준: **2026-09-18**  
> 책의 학습 목표는 유지하면서 LangChain 1.x의 분리된 provider 패키지와 현재 메시지/스트리밍 API에 맞췄습니다. 모델 이름은 공급자 정책에 따라 바뀔 수 있으므로 환경 변수로 덮어쓸 수 있게 구성했습니다.


In [ ]:
%pip install -qU langchain-huggingface transformers torch accelerate huggingface_hub python-dotenv


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
cache_dir = Path("cache/huggingface")
cache_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(cache_dir.resolve()))

model_id = os.getenv("HF_LOCAL_MODEL", "HuggingFaceTB/SmolLM2-1.7B-Instruct")


## 1. `from_model_id`로 빠르게 로드


In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

hf_llm = HuggingFacePipeline.from_model_id(
    model_id=model_id,
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 256,
        "do_sample": False,
        "repetition_penalty": 1.03,
        "return_full_text": False,
    },
)
chat = ChatHuggingFace(llm=hf_llm)
print(chat.invoke("대한민국의 수도는 어디인가요?").text)


## 2. Transformers pipeline을 직접 구성

모델·토크나이저 로딩 옵션을 세밀하게 제어할 때 사용합니다.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
)

text_generation = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,
    repetition_penalty=1.03,
    return_full_text=False,
)
custom_llm = HuggingFacePipeline(
    pipeline=text_generation,
    model_id=model_id,
)
custom_chat = ChatHuggingFace(llm=custom_llm)
print(custom_chat.invoke("RAG를 두 문장으로 설명해 주세요.").text)


## Gated 모델

먼저 모델 페이지에서 라이선스에 동의하고 `.env`에 `HUGGINGFACEHUB_API_TOKEN`을 설정합니다. 토큰을 코드에 직접 입력하지 않습니다.


In [ ]:
gated_model_id = os.getenv("HF_GATED_MODEL", "google/gemma-3-4b-it")
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

if not hf_token:
    print("Gated 모델 예제를 실행하려면 HUGGINGFACEHUB_API_TOKEN을 설정하세요.")
else:
    gated_tokenizer = AutoTokenizer.from_pretrained(
        gated_model_id, token=hf_token
    )
    gated_model = AutoModelForCausalLM.from_pretrained(
        gated_model_id,
        token=hf_token,
        torch_dtype="auto",
        device_map="auto",
    )
    gated_pipe = pipeline(
        "text-generation",
        model=gated_model,
        tokenizer=gated_tokenizer,
        max_new_tokens=256,
        do_sample=False,
        return_full_text=False,
    )
    gated_chat = ChatHuggingFace(
        llm=HuggingFacePipeline(
            pipeline=gated_pipe,
            model_id=gated_model_id,
        )
    )
    print(gated_chat.invoke("대한민국의 수도는 어디인가요?").text)


## LCEL과 배치 실행


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "질문에 한 문장으로 답하세요."),
        ("human", "{question}"),
    ]
)
chain = prompt | chat | StrOutputParser()

questions = [
    {"question": "숫자 1을 한글로 쓰면?"},
    {"question": "숫자 2를 한글로 쓰면?"},
    {"question": "숫자 3을 한글로 쓰면?"},
    {"question": "숫자 4를 한글로 쓰면?"},
]
for answer in chain.batch(questions, config={"max_concurrency": 2}):
    print(answer)


### GPU 메모리 팁

- 단일 GPU를 강제할 때는 `device=0`을 사용합니다.
- 여러 장치에 자동 배치하려면 `device_map="auto"`와 `accelerate`를 사용합니다.
- 두 옵션을 동시에 지정하지 않습니다.
- 생성 옵션은 `pipeline_kwargs` 또는 Transformers `pipeline()`에, 모델 로딩 옵션은 `model_kwargs`/`from_pretrained()`에 둡니다.
